## 📦 Cell 1 — Library Imports

Every external tool must be `import`ed before use. This cell loads the full toolkit for data analysis, visualisation, and machine learning.

| Library / Module | Alias | Purpose |
|---|---|---|
| `numpy` | `np` | Fast numerical arrays, math operations, data types |
| `pandas` | `pd` | DataFrames — the main structure for loading and manipulating tabular data |
| `matplotlib.pyplot` | `plt` | Low-level plotting engine: creates figures, axes, and chart primitives |
| `seaborn` | `sns` | High-level statistical charts (boxplots, heatmaps, histplots) built on matplotlib |
| `os` | — | OS utilities: file paths, directory listing |
| `scatter_matrix` | — | Plots every pair of numeric features against each other (pairplot) |
| `LabelEncoder` | — | Converts string categories → integers (e.g. "Yes"→1, "No"→0) |
| `MinMaxScaler` | — | Scales numeric features to a [0, 1] range |
| `train_test_split` | — | Randomly splits data into training and test sets |
| `LinearRegression` | — | Ordinary least-squares straight-line regression model |
| `r2_score` | — | R² metric: proportion of variance explained by the model (0–1, higher = better) |
| `mean_absolute_error` | — | MAE: average absolute difference between predictions and true values |
| `mean_absolute_percentage_error` | — | MAPE: MAE expressed as a percentage of the true values |
| `mean_squared_error` | — | MSE: average squared difference — penalises large errors more than MAE |
| `DecisionTreeRegressor` | — | Single decision tree model for regression |
| `BaggingRegressor` | — | Bootstrap aggregating: trains many models on random subsets and averages them |
| `AdaBoostRegressor` | — | Adaptive boosting: each model corrects the previous model's errors |
| `RandomForestRegressor` | — | Ensemble of decision trees with random feature selection at each split |


In [ ]:
import numpy as np                    # Numerical operations and array support
import pandas as pd                   # DataFrames for data loading, cleaning, and manipulation
import matplotlib.pyplot as plt       # Base plotting library for figures and axes
import seaborn as sns                 # High-level statistical visualisation library
import os                             # Operating system interface (paths, directories)

from pandas.plotting import scatter_matrix          # Pairwise scatter plots across all numeric features
from sklearn.preprocessing import LabelEncoder      # Encode text labels as integer codes
from sklearn.preprocessing import MinMaxScaler       # Scale features to [0, 1] range
from sklearn.model_selection import train_test_split # Random train/test split
from sklearn.linear_model import LinearRegression    # Ordinary least-squares regression
from sklearn.metrics import r2_score                 # R² coefficient of determination
from sklearn.metrics import (                        # Three regression error metrics
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error
)

from pandas.plotting import scatter_matrix           # Duplicate import (safe but unnecessary)
from sklearn.tree import DecisionTreeRegressor       # Single decision-tree regressor
from sklearn.ensemble import BaggingRegressor        # Bootstrap aggregating ensemble
from sklearn.ensemble import AdaBoostRegressor       # Adaptive boosting ensemble
from sklearn.ensemble import RandomForestRegressor   # Random Forest ensemble


## 📂 Cell 2 — Load the Dataset

`pd.read_csv()` reads a comma-separated file from disk into a **DataFrame** — a 2-D table with labelled rows and columns.

`df.head()` shows the **first 5 rows**. This is the quickest sanity check right after loading:
- Did the file parse correctly?
- Are column names on the correct row?
- Does the data look as expected?

> The dataset represents home maintenance technicians in Egypt with attributes like service type, experience, ratings, reviews, pricing (EGP), and availability.


In [ ]:
df = pd.read_csv('home_maintenance_technicians.csv')  # Load the CSV into a DataFrame named 'df'

df.head()  # Display the first 5 rows to confirm the file loaded correctly


## 🔍 Cell 3 — Initial Data Quality Audit

This cell performs four key inspection steps in sequence:

### 1. `df.isna().sum()` — Missing Value Count
`df.isna()` produces a Boolean DataFrame — `True` wherever a value is missing. `.sum()` collapses each column, counting the `True` values. Result: missing count per column.

### 2. `df.duplicated().sum()` — Duplicate Row Count
`df.duplicated()` returns `True` for any row that is an exact copy of an earlier row. `.sum()` counts them. Duplicates distort statistics and can cause data leakage in ML.

### 3. `df.describe()` — Descriptive Statistics
Provides for every numeric column:

| Statistic | Meaning |
|---|---|
| `count` | Non-null values |
| `mean` | Average |
| `std` | Spread around the mean |
| `min` / `max` | Range boundaries |
| `25%`, `50%`, `75%` | Quartiles — describe the distribution shape |

**Red flags:** negative values where impossible, `count` lower than total rows (hidden NaNs), extreme `max` vs `75%` gaps (outliers).

### 4. `df.info()` — Schema Summary
Prints column names, non-null counts, data types (`int64`, `float64`, `object`), and memory usage. If a numeric column shows `object` dtype, numbers were stored as strings and need converting before modelling.

`display()` renders the output as a formatted HTML table in Jupyter (better than `print()` for DataFrames).


In [ ]:
print("sum of nulls :\n ")        # Header label
print(df.isna().sum())             # Count missing values per column

print("\nsum of duplicated :\n ")   # Header label
print(df.duplicated().sum())       # Count exact duplicate rows

print("\nStatistical Summary:\n")   # Header label
display(df.describe())             # Show count, mean, std, min, quartiles, max for numerics

print("\ninformation Summary:\n")   # Header label
display(df.info())                 # Show column names, dtypes, non-null counts, memory


## 🔢 Cell 4 — Define Numeric Feature List

A Python **list** named `num_cols` is defined, containing the names of the five numeric columns of interest.

**Why define this list explicitly instead of using `select_dtypes()`?**
- It documents *intentionality* — you have consciously chosen these five columns for analysis
- `technician_id` is technically numeric but is an identifier, not a measurement — excluding it here avoids meaningless statistics on an ID number
- The list is reused in the next several cells (histograms, boxplots, heatmap), keeping the code DRY (Don't Repeat Yourself)

| Column | Meaning |
|---|---|
| `experience_years` | Years of professional experience |
| `rating` | Average customer rating (e.g. 1–5 stars) |
| `number_of_reviews` | Total number of customer reviews |
| `base_price_EGP` | Service price in Egyptian Pounds — the **target variable** |
| `completed_jobs` | Total jobs completed |


In [ ]:
num_cols = [           # Define a list of the five numeric columns to analyse
    'experience_years',  # Years of professional experience
    'rating',            # Average customer star rating
    'number_of_reviews', # Count of customer reviews
    'base_price_EGP',    # Service price in EGP — the target variable we want to predict
    'completed_jobs'     # Total number of completed jobs
]


## 📊 Cell 5 — Histograms of All Numeric Columns

### What this visualisation shows
A **histogram** divides a column's range into equal-width buckets (`bins`) and counts how many values fall into each one. The x-axis is the value range; the y-axis is frequency (count). This reveals the **distribution shape** of each variable.

### How to read it
| Pattern | What it means |
|---|---|
| Bell-shaped (normal) | Values cluster symmetrically around the mean |
| Right-skewed (long right tail) | Most values are low; a few are very high — common for prices and review counts |
| Left-skewed (long left tail) | Most values are high; a few are very low |
| Flat (uniform) | Values are spread evenly — no strong central tendency |
| Bimodal (two peaks) | Two distinct groups exist in the data |

### Code explanation
- `df[num_cols].hist(figsize=(15, 8), bins=20)` — calls pandas' built-in histogram method on the sub-DataFrame of the five numeric columns. `figsize=(15,8)` sets the figure width and height in inches. `bins=20` means each column's range is divided into 20 equal-width buckets
- `plt.tight_layout()` — automatically adjusts subplot spacing so titles and labels don't overlap
- `plt.show()` — renders the figure to the screen and clears the plot buffer


In [ ]:
df[num_cols].hist(figsize=(15, 8), bins=20)  # Plot a histogram for each of the 5 numeric columns
                                              # figsize=(15,8): figure is 15 wide × 8 inches tall
                                              # bins=20: divide the value range into 20 equal buckets

plt.tight_layout()  # Auto-adjust spacing between subplots to prevent label overlap
plt.show()          # Render the figure and clear the plot buffer


## 📦 Cell 6 — Boxplots for Outlier Detection

### What this visualisation shows
A **boxplot** (also called a box-and-whisker plot) summarises a column's distribution using five statistics:

```
|-----whisker-----|[  Q1 |====median====| Q3  ]|-----whisker-----|
                               ↑
                          Interquartile
                          Range (IQR)
                       = Q3 − Q1 (the box)
```

| Element | Meaning |
|---|---|
| **Box** | Spans Q1 (25th percentile) to Q3 (75th percentile) — the middle 50% of data |
| **Line inside box** | Median (50th percentile) |
| **Whiskers** | Extend to Q1 − 1.5×IQR and Q3 + 1.5×IQR |
| **Dots beyond whiskers** | **Outliers** — statistically extreme values |

Boxplots are the primary visual tool for outlier detection. If you see many dots far from the whiskers, that column has outliers worth investigating.

### Code explanation
- `plt.figure(figsize=(15, 6))` — creates a new blank figure (canvas) 15×6 inches
- `for i, col in enumerate(num_cols, 1)` — loops over the column list; `enumerate(..., 1)` gives `(1, col1), (2, col2), ...` (starting at 1, not 0, because `plt.subplot` indices start at 1)
- `plt.subplot(2, 3, i)` — divides the figure into a 2-row × 3-column grid and activates position `i`
- `sns.boxplot(y=df[col])` — draws a vertical boxplot for the column. Using `y=` makes it vertical (values on the y-axis)
- `plt.title(col)` — labels each subplot with the column name


In [ ]:
plt.figure(figsize=(15, 6))  # Create a 15×6 inch figure to hold all 5 boxplots

for i, col in enumerate(num_cols, 1):  # Loop: i=1..5, col=each column name
                                        # enumerate(start=1) because subplot indices start at 1
    plt.subplot(2, 3, i)               # Activate the i-th cell in a 2-row × 3-col grid
    sns.boxplot(y=df[col])             # Draw vertical boxplot — outliers appear as dots beyond whiskers
    plt.title(col)                     # Label this subplot with the column name

plt.tight_layout()  # Fix spacing so titles don't overlap between subplots
plt.show()          # Render and clear


## 🌡️ Cell 7 — Correlation Heatmap

### What this visualisation shows
A **correlation heatmap** displays the **Pearson correlation coefficient** between every pair of numeric columns. Each cell contains a value from **−1 to +1**:

| Value | Interpretation |
|---|---|
| **+1.0** | Perfect positive correlation — as X increases, Y increases proportionally |
| **+0.5 to +1.0** | Strong positive correlation |
| **0** | No linear relationship |
| **−0.5 to −1.0** | Strong negative correlation — as X increases, Y decreases |
| **−1.0** | Perfect negative correlation |

The colour scale (`coolwarm`) maps positive correlations to **red/warm** tones and negative correlations to **blue/cool** tones. The diagonal is always 1.0 (every variable is perfectly correlated with itself).

### Why this matters for ML
- **High correlation with the target (`base_price_EGP`)** → that feature is likely a useful predictor
- **High correlation between two features** → multicollinearity; in linear regression, one can be dropped without losing information
- **Near-zero correlation with target** → that feature may not help the model and could add noise

### Code explanation
- `df[num_cols].corr()` — computes the pairwise Pearson correlation matrix (a 5×5 DataFrame)
- `sns.heatmap(..., annot=True, cmap='coolwarm')` — `annot=True` prints the numeric value in each cell; `cmap='coolwarm'` sets the colour palette
- `plt.title(...)` — adds a title above the heatmap


In [ ]:
plt.figure(figsize=(8, 6))  # Create an 8×6 inch figure for the heatmap

sns.heatmap(
    df[num_cols].corr(),  # Compute 5×5 Pearson correlation matrix, then pass to heatmap
    annot=True,           # Print the correlation value (e.g. 0.72) inside each cell
    cmap='coolwarm'       # Red = positive correlation, Blue = negative correlation
)

plt.title('Correlation Matrix')  # Add a descriptive title above the heatmap
plt.show()                       # Render the figure


## 📈 Cell 8 — Distribution Plots with KDE (Features vs Target)

### What this visualisation shows
`sns.histplot(..., kde=True)` combines two things on the same axes:
1. A **histogram** (bars) showing the raw frequency of values in each bin
2. A **KDE curve** (Kernel Density Estimate — the smooth line) — a continuous, smoothed estimate of the underlying probability distribution

### How to read the KDE curve
The KDE curve rises where data is dense and falls where it is sparse. Unlike the histogram (which changes shape with `bins`), the KDE is stable and shows the true shape of the distribution:
- A single peak → unimodal distribution
- A long tail to the right → right-skewed (common for prices and counts)
- A smooth bell → approximately normal

### Why these four features?
These are the four **predictor variables** (features) used in the model — all except `base_price_EGP` (the target). Visualising their distributions shows whether normalisation or transformation is needed before modelling.

### Code explanation
- `num_features` — the four predictor columns (excludes `base_price_EGP`)
- `plt.subplot(2, 2, i)` — a 2×2 grid for 4 plots
- `sns.histplot(df[col], kde=True, bins=20, color='red')` — red-coloured histogram with KDE overlay, 20 bins
- `plt.title(f'{col} vs base_price_EGP')` — f-string title; note the title says "vs base_price_EGP" to flag these as predictors relative to the target


In [ ]:
num_features = [           # The four numeric predictor columns (target excluded)
    'experience_years',
    'rating',
    'number_of_reviews',
    'completed_jobs'
]

plt.figure(figsize=(15, 10))  # Create a 15×10 inch figure for the 2×2 grid

for i, col in enumerate(num_features, 1):     # Loop: i=1..4, col=each feature name
    plt.subplot(2, 2, i)                       # Activate the i-th cell in a 2-row × 2-col grid
    sns.histplot(                              # Draw histogram + KDE for this feature
        df[col],                               # The data Series for this column
        kde=True,                              # Overlay a Kernel Density Estimate curve
        bins=20,                               # 20 histogram bins
        color='red'                            # Use red fill for the histogram bars
    )
    plt.title(f'{col} vs base_price_EGP')      # Label each subplot (f-string inserts column name)

plt.tight_layout()  # Prevent subplot titles and labels from overlapping
plt.show()          # Render and clear


## 📦 Cell 9 — Boxplot: Price Distribution by Service Type

### What this visualisation shows
This boxplot compares the **distribution of `base_price_EGP`** across each **service type** (e.g. plumbing, electrical, painting). It answers the question:

> *Does the type of service significantly affect the price a technician charges?*

Each box represents one service category. You can immediately see:
- Which service types command the **highest prices** (highest median)
- Which have the **widest price variation** (tallest box / long whiskers)
- Whether any service type has many **outlier prices** (dots beyond whiskers)
- Whether price ranges **overlap** between services (if they don't, service type is a strong predictor)

### Code explanation
- `sns.boxplot(data=df, x='service_type', y='base_price_EGP')` — `x` groups the boxes by service category; `y` is the numeric variable being compared
- `plt.xticks(rotation=45)` — rotates x-axis category labels 45 degrees to prevent overlap when there are many long category names


In [ ]:
plt.figure(figsize=(10, 5))  # Create a 10×5 inch figure

sns.boxplot(
    data=df,                  # The full DataFrame
    x='service_type',         # Group boxes by service type (one box per category)
    y='base_price_EGP'        # The numeric variable to compare across groups
)

plt.xticks(rotation=45)  # Rotate x-axis labels 45° to prevent text overlap
plt.show()               # Render and clear


## 📦 Cell 10 — Boxplot: Price Distribution by Availability Status

### What this visualisation shows
This boxplot compares `base_price_EGP` across **availability statuses** (e.g. "Available", "Busy", "On Leave"). It answers:

> *Do technicians who are currently available price their services differently from those who are busy or on leave?*

This might reveal economic behaviour — for example, highly sought-after (busy) technicians may charge higher prices, or available technicians may offer lower prices to attract customers.

- **Median line position** → typical price for each status
- **Box height** → price variability within each status
- **Overlapping boxes** → no strong price difference between statuses
- **Non-overlapping boxes** → availability status is likely a meaningful predictor of price

### Code explanation
Same structure as the previous boxplot, but with `x='availability_status'`. No `plt.xticks(rotation=45)` is needed here because availability statuses tend to have shorter labels.


In [ ]:
plt.figure(figsize=(10, 5))  # Create a 10×5 inch figure

sns.boxplot(
    data=df,                   # The full DataFrame
    x='availability_status',   # Group boxes by availability status
    y='base_price_EGP'         # Compare price distributions across each status
)

plt.show()  # Render and clear


## 📉 Cell 11 — Line Plot: Rating vs Price

### What this visualisation shows
`sns.lineplot(x='rating', y='base_price_EGP')` draws a line connecting the **mean `base_price_EGP`** at each distinct `rating` value, with a shaded confidence interval band around it.

It visually answers:

> *Does a higher customer rating correlate with a higher service price?*

### How to read it
- **Rising line** → higher-rated technicians charge more (positive relationship)
- **Flat line** → rating has little effect on price
- **Falling line** → higher-rated technicians may undercut on price to attract volume
- **Wide confidence band** → high variance at that rating level (fewer data points or large spread)
- **Narrow band** → consistent prices at that rating level

> **Note:** `lineplot` by default **aggregates** — it computes the mean of `y` at each unique `x` value and draws the line through those means. It does not simply connect raw data points.

### Code explanation
- `sns.lineplot(data=df, x='rating', y='base_price_EGP')` — seaborn reads column names directly from `data=df`


In [ ]:
plt.figure(figsize=(8, 5))  # Create an 8×5 inch figure

sns.lineplot(
    data=df,               # Source DataFrame
    x='rating',            # X-axis: customer rating value
    y='base_price_EGP'     # Y-axis: mean price at each rating level (with confidence band)
)

plt.show()  # Render and clear


## 🎯 Cell 12 — Correlation of All Numeric Features with Target

### What this does
This cell computes and prints the **Pearson correlation coefficient** between every numeric column and the target variable `base_price_EGP`, sorted from strongest to weakest.

### Why it matters
This is **feature importance by linear correlation** — the fastest way to identify which numeric variables have the strongest linear relationship with the target *before* running any model.

- Values near **+1** → strong positive predictor
- Values near **−1** → strong negative predictor
- Values near **0** → little to no linear relationship with price

> **Important caveat:** Pearson correlation only captures *linear* relationships. A feature could have a strong non-linear relationship with the target and still show near-zero correlation here. This is why tree-based models (Random Forest, Gradient Boosting) often find features that linear correlation misses.

### Code explanation
- `df.select_dtypes(include=['number'])` — select all numeric columns (avoids errors from passing strings to `.corr()`)
- `.corr()` — computes the full correlation matrix
- `corr['base_price_EGP']` — extract just the column of correlations with the target
- `.sort_values(ascending=False)` — sort from highest (most positively correlated) to lowest


In [ ]:
# Correlation with target

corr = df.select_dtypes(include=['number']).corr()  # Compute correlation matrix for all numeric columns

print(
    corr['base_price_EGP']              # Extract correlations with the target variable
    .sort_values(ascending=False)       # Sort: strongest positive correlation at the top
)


## 🗑️ Cell 13 — Drop Non-Predictive Identifier Columns

Two columns are dropped before modelling:

| Column | Why dropped |
|---|---|
| `technician_id` | A unique identifier — carries no predictive signal; just an arbitrary number |
| `technician_name` | A free-text name — too many unique values, not useful as a categorical feature |

`df.drop(columns=[...])` returns a **new DataFrame** with those columns removed. The result is stored in `df_model` — a clean copy used for all modelling steps, while the original `df` is preserved for reference.

> **Best practice:** Never modify `df` in place during modelling. Keep a clean original and work on a copy (`df_model`). This makes it easy to restart modelling without reloading the data.


In [ ]:
df_model = df.drop(columns=[
    'technician_id',    # Drop the unique ID — an arbitrary number with no predictive value
    'technician_name'   # Drop the name — too many unique text values to use as a feature
])                      # Result stored in df_model; original df is unchanged


## ✂️ Cell 14 — Separate Features (X) and Target (y)

In supervised machine learning, the dataset is always split into two parts:

- **`X` (features / predictors)** — the input columns the model uses to make predictions. Created by dropping the target column from `df_model`
- **`y` (target / label)** — the single column the model is trying to predict (`base_price_EGP`)

`df_model.drop('base_price_EGP', axis=1)` — `axis=1` means "drop a column" (axis=0 would drop a row).

This separation is required by every scikit-learn model — they always expect `model.fit(X_train, y_train)`.


In [ ]:
X = df_model.drop('base_price_EGP', axis=1)  # Features: all columns except the target
                                              # axis=1 means drop a column (not a row)

y = df_model['base_price_EGP']               # Target: the single column to predict


## 🔡 Cell 15 — One-Hot Encode Categorical Columns

### The problem
Machine learning models work with **numbers only** — they cannot process raw text like `"plumbing"` or `"Available"`. Categorical columns must be converted to numeric form.

### The solution — One-Hot Encoding (`pd.get_dummies`)
One-Hot Encoding creates a **new binary column for each unique category**:

```
service_type       →    service_type_electrical  service_type_painting  service_type_plumbing
"electrical"              1                        0                      0
"painting"                0                        1                      0
"plumbing"                0                        0                      1
```

Each row gets a `1` in its category's column and `0` everywhere else.

### `drop_first=True` — Why?
With `k` categories, only `k-1` columns are needed — the last category is implied when all others are `0`. For example, if `service_type_electrical=0` and `service_type_painting=0`, then it must be plumbing. Dropping one column:
- Saves memory
- Removes **multicollinearity** (a problem for linear regression when two columns are perfectly predictable from each other)

### Code explanation
- `pd.get_dummies(X, drop_first=True)` — applies one-hot encoding to all `object` dtype columns in `X` automatically
- `print(X.shape)` — confirms the new number of columns (will be higher than before, one column per category minus one)
- `X.head()` — shows the transformed feature matrix with the new binary columns


In [ ]:
X = pd.get_dummies(      # Apply one-hot encoding to all categorical (object) columns in X
    X,
    drop_first=True        # Drop the first category per feature to avoid multicollinearity
)                          # Result: new binary columns for each category (True/False → 1/0)

print("Shape of X:", X.shape)  # Print new dimensions — columns increased after encoding
print("Shape of y:", y.shape)  # Print target shape — should match number of rows in X

display(X.head())              # Preview the encoded feature matrix


## ✂️ Cell 16 — Train / Test Split

### Why split the data?
A model trained and evaluated on the **same data** will appear to perform well even if it has simply memorised the data rather than learned generalizable patterns (**overfitting**).

By holding out a **test set** that the model never sees during training, we get an honest estimate of how the model will perform on new, unseen data.

### How `train_test_split` works
It randomly shuffles the data and splits it into two non-overlapping subsets:

```
All data (100%)
├── Training set (80%) → model learns from this
└── Test set    (20%) → model is evaluated on this
```

### Parameters
- `test_size=0.2` → 20% of rows go to the test set; 80% to training
- `random_state=42` → fixes the random seed so the same split is produced every run (reproducibility). The number `42` is a convention but any integer works.

### Output
Four objects are created:
- `X_train` — training features
- `X_test` — test features
- `y_train` — training target values
- `y_test` — test target values (the ground truth to compare predictions against)


In [ ]:
from sklearn.model_selection import train_test_split  # Import the splitter

# Split X and y simultaneously — rows are aligned so train rows of X match train rows of y
X_train, X_test, y_train, y_test = train_test_split(
    X,               # Feature matrix
    y,               # Target vector
    test_size=0.2,   # Reserve 20% of data for the test set
    random_state=42  # Fix random seed for reproducible splits across runs
)

print(X_train.shape)  # e.g. (800, 15) → 800 training rows, 15 features
print(X_test.shape)   # e.g. (200, 15) → 200 test rows, 15 features


## 📏 Cell 17 — Model 1: Linear Regression

### What is Linear Regression?
Linear Regression fits a **straight-line (hyperplane) relationship** between features and target:

```
price = w₁×experience + w₂×rating + w₃×reviews + ... + bias
```

It finds the weights (`w`) that **minimise the sum of squared residuals** (OLS — Ordinary Least Squares). It is the simplest regression model and serves as the **baseline** — if a complex model doesn't beat it significantly, the simpler model is preferred.

### Evaluation Metrics

| Metric | Formula | What it measures |
|---|---|---|
| **MAE** | mean(\|y_true − y_pred\|) | Average absolute error in EGP — easy to interpret |
| **RMSE** | √MSE | Root Mean Squared Error — penalises large errors more than MAE |
| **R²** | 1 − SS_res/SS_tot | Proportion of variance explained (0 = no better than mean, 1 = perfect) |

### The four-step scikit-learn pattern
1. **Instantiate** — `LinearRegression()` creates the model object
2. **Train** — `lr.fit(X_train, y_train)` learns the weights from training data
3. **Predict** — `lr.predict(X_test)` generates predictions for test inputs
4. **Evaluate** — compare `y_pred_lr` against `y_test` using metrics

> **RMSE = √MSE** — taking the square root returns the error to the original unit (EGP), making it more interpretable than MSE which is in EGP².


In [ ]:
from sklearn.linear_model import LinearRegression  # Import the model
from sklearn.metrics import (                       # Import evaluation metrics
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

lr = LinearRegression()          # Instantiate the model (no hyperparameters needed)

lr.fit(X_train, y_train)         # Train: learn weights that minimise squared errors on training data

y_pred_lr = lr.predict(X_test)   # Predict: apply learned weights to test features

mae = mean_absolute_error(       # MAE: average |true - predicted| in EGP
    y_test,
    y_pred_lr
)

mse = mean_squared_error(        # MSE: average (true - predicted)² — larger errors penalised more
    y_test,
    y_pred_lr
)

rmse = mse ** 0.5                # RMSE: square root of MSE — back in original EGP units

r2 = r2_score(                   # R²: proportion of variance in price explained by the model
    y_test,
    y_pred_lr
)

print("Linear Regression Results")
print("-" * 30)
print("MAE :", mae)        # Print Mean Absolute Error
print("RMSE:", rmse)       # Print Root Mean Squared Error
print("R2 Score:", r2)     # Print R² score


## 🌲 Cell 18 — Model 2: Random Forest Regressor

### What is Random Forest?
Random Forest is an **ensemble of decision trees**. Each tree is trained on a **random bootstrap sample** of the training data (sampling with replacement), and at each split, only a **random subset of features** is considered.

The final prediction is the **average** of all 100 individual tree predictions.

### Why is it better than a single tree?
A single decision tree is high-variance — small changes in training data produce very different trees (**overfitting**). By averaging many trees trained on different data and features, the variance cancels out while the bias stays low. This is the core idea of **bagging** (Bootstrap AGGregatING).

### Key hyperparameters
- `n_estimators=100` — the number of trees in the forest. More trees = more stable predictions (diminishing returns after ~100–200)
- `random_state=42` — fixes the random seed for reproducibility

### Advantage over Linear Regression
- Handles **non-linear relationships** automatically (no need to manually add polynomial features)
- Robust to **outliers** (trees split on thresholds, not affected by extreme values)
- Automatically handles **feature interactions** (e.g., high experience AND high rating together → premium price)
- Does **not require feature scaling** (tree splits are threshold-based, not distance-based)


In [ ]:
from sklearn.ensemble import RandomForestRegressor  # Import the Random Forest model

rf = RandomForestRegressor(    # Instantiate the model with hyperparameters
    n_estimators=100,           # Build 100 decision trees in the forest
    random_state=42             # Fix random seed for reproducible results
)

rf.fit(X_train, y_train)       # Train: each of the 100 trees learns on a bootstrap sample

y_pred_rf = rf.predict(X_test) # Predict: average the predictions of all 100 trees

mae_rf = mean_absolute_error(  # MAE for Random Forest predictions
    y_test,
    y_pred_rf
)

mse_rf = mean_squared_error(   # MSE for Random Forest predictions
    y_test,
    y_pred_rf
)

rmse_rf = mse_rf ** 0.5        # RMSE: back in EGP units

r2_rf = r2_score(              # R² for Random Forest
    y_test,
    y_pred_rf
)

print("Random Forest Results")
print("-" * 30)
print("MAE :", mae_rf)       # Print MAE
print("RMSE:", rmse_rf)      # Print RMSE
print("R2 Score:", r2_rf)    # Print R²


## 🚀 Cell 19 — Model 3: Gradient Boosting Regressor

### What is Gradient Boosting?
Gradient Boosting builds trees **sequentially** — each new tree is trained to **correct the residual errors** of all previous trees combined:

```
Tree 1 → predicts price (makes errors)
Tree 2 → predicts the residual errors of Tree 1
Tree 3 → predicts the residual errors of Trees 1+2
...
Final prediction = weighted sum of all tree predictions
```

This is the key difference from Random Forest:
- **Random Forest** — trees are independent, trained in parallel, averaged
- **Gradient Boosting** — trees are sequential, each focused on what the previous ones got wrong

### Key hyperparameters
- `n_estimators=100` — number of trees (boosting rounds)
- `learning_rate=0.1` — how much each new tree's contribution is shrunk before adding it. Lower = more conservative, less risk of overfitting, needs more trees. Typical range: 0.01–0.3
- `random_state=42` — reproducibility seed

### Trade-offs vs Random Forest
| | Random Forest | Gradient Boosting |
|---|---|---|
| Speed | Faster (parallel) | Slower (sequential) |
| Accuracy | Very good | Often slightly better |
| Overfitting risk | Lower | Higher (sensitive to `learning_rate`) |
| Hyperparameter tuning | Less critical | More important |


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor  # Import the Gradient Boosting model

gbr = GradientBoostingRegressor(  # Instantiate with hyperparameters
    n_estimators=100,              # Number of sequential boosting rounds (trees)
    learning_rate=0.1,             # Shrinkage factor: scales each tree's contribution (0.01–0.3 typical)
    random_state=42                # Fix random seed for reproducibility
)

gbr.fit(X_train, y_train)         # Train: each tree corrects the residuals of all previous trees

y_pred_gbr = gbr.predict(X_test)  # Predict: weighted sum of all 100 trees

mae_gbr = mean_absolute_error(    # MAE for Gradient Boosting predictions
    y_test,
    y_pred_gbr
)

mse_gbr = mean_squared_error(     # MSE for Gradient Boosting predictions
    y_test,
    y_pred_gbr
)

rmse_gbr = mse_gbr ** 0.5         # RMSE: back in EGP units

r2_gbr = r2_score(                # R² for Gradient Boosting
    y_test,
    y_pred_gbr
)

print("Gradient Boosting Results")
print("-" * 30)
print("MAE :", mae_gbr)       # Print MAE
print("RMSE:", rmse_gbr)      # Print RMSE
print("R2 Score:", r2_gbr)    # Print R²


## 🔄 Cell 20 — Cross-Validation (5-Fold CV)

### Why cross-validation?
A single train/test split can produce **lucky or unlucky results** depending on which rows ended up in the test set. If the test set happened to contain only easy-to-predict rows, R² looks great — but this doesn't reflect real performance.

**K-Fold Cross-Validation** gives a more reliable estimate by evaluating the model `k` times on different data subsets:

```
Fold 1: [TEST | train | train | train | train] → R² score 1
Fold 2: [train | TEST | train | train | train] → R² score 2
Fold 3: [train | train | TEST | train | train] → R² score 3
Fold 4: [train | train | train | TEST | train] → R² score 4
Fold 5: [train | train | train | train | TEST] → R² score 5
                                                  ↓
                                           Mean R² ± std
```

Every row is used for testing exactly once. The mean score is a more robust estimate of generalisation performance than a single split.

### Code explanation
- `cross_val_score(lr, X_train, y_train, cv=5, scoring='r2')` — runs 5-fold CV on the **training set only** using Linear Regression
  - `cv=5` → 5 folds
  - `scoring='r2'` → evaluate each fold using R²
- `scores` — a numpy array of 5 R² values (one per fold)
- `scores.mean()` — the average R² across all folds — the single number to report as model performance

> **Important:** Cross-validation is run on `X_train`/`y_train` only — never on the test set. The test set is kept completely separate as a final holdout.


In [ ]:
from sklearn.model_selection import cross_val_score  # Import the CV utility
from sklearn.linear_model import LinearRegression    # Re-import to be explicit (already imported above)

lr = LinearRegression()  # Instantiate a fresh Linear Regression model for CV

scores = cross_val_score(  # Run k-fold cross-validation
    lr,                    # The model to evaluate
    X_train,               # Features — CV splits are made within training data only
    y_train,               # Target — corresponding labels for the training rows
    cv=5,                  # 5 folds: data is split into 5 parts; each takes a turn as test set
    scoring='r2'           # Evaluate each fold using the R² metric
)                          # Returns a numpy array of 5 R² scores

print("CV R2 Scores:", scores)    # Print all 5 individual fold scores
print("Mean R2:", scores.mean())  # Print the average — the robust performance estimate


## 📋 Cell 21 — Final Model Summary Report

This cell prints a consolidated performance summary for the **Linear Regression** model (used as the primary baseline), comparing its test-set metrics against its cross-validation mean R².

### What to compare

| Metric | Source | What it tells you |
|---|---|---|
| **Test R²** | Single train/test split | Model's fit on the held-out test set |
| **Test MAE** | Single train/test split | Average prediction error in EGP |
| **Test RMSE** | Single train/test split | Error in EGP, penalising large mistakes more |
| **CV Mean R²** | 5-fold cross-validation | More reliable estimate of generalisation |

### Key diagnostic
- If **Test R² ≈ CV Mean R²** → the test split was representative; results are trustworthy
- If **Test R² >> CV Mean R²** → the test split was lucky; CV is the more honest number
- If **Test R² << CV Mean R²** → the test split was unlucky; model is actually better than it looks on the test set

A good model has **high R²** (close to 1), **low MAE and RMSE**, and **small gap** between test and CV scores.


In [ ]:
print("FINAL SUMMARY")
print("-" * 30)

print("Test R²:", r2)      # R² from the single held-out test set
print("Test MAE:", mae)    # Mean Absolute Error on test set (in EGP)
print("Test RMSE:", rmse)  # Root Mean Squared Error on test set (in EGP)

print("\nCross Validation Mean R²:", scores.mean())  # Average R² across 5 CV folds — more reliable
